# 5.2 Tracking des travailleurs (Axe D)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import torch

RUNS_DIR    = Path('/root/Projet_Image/runs')
BEST_WEIGHTS = RUNS_DIR / 'yolov8m_epi_1024-2' / 'weights' / 'best.pt'
OUTPUT_DIR  = Path('/root/Projet_Image/tracking_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(str(BEST_WEIGHTS))
print(f'Modèle chargé : {BEST_WEIGHTS}')
print(f'Device        : {DEVICE}')

## 1. Règle de conformité

Trois EPI sont disponibles : casque, gilet de sécurité et gants — comme dans l'application Streamlit (Axe E).

Les gants sont désactivés par défaut (`VERIFIER_GANTS = False`) : la classe `hands` est peu détectée en vidéo (mains souvent hors-champ ou en mouvement rapide), ce qui génère des fausses alertes en continu. Passer la variable à `True` pour les activer.

In [ ]:
from collections import defaultdict, deque

# Mettre True pour activer la vérification des gants
# (désactivé par défaut — hands peu détectée en vidéo → fausses alertes permanentes)
VERIFIER_GANTS = False

REGLES_CONFORMITE = {
    'helmet':      {'partie_corps': 'head',   'message': 'casque manquant'},
    'safety-vest': {'partie_corps': 'person', 'message': 'gilet de securite manquant'},
}
if VERIFIER_GANTS:
    REGLES_CONFORMITE['gloves'] = {'partie_corps': 'hands', 'message': 'gants manquants'}


def iou(boite_a, boite_b):
    xa1, ya1, xa2, ya2 = boite_a
    xb1, yb1, xb2, yb2 = boite_b
    x1, y1 = max(xa1, xb1), max(ya1, yb1)
    x2, y2 = min(xa2, xb2), min(ya2, yb2)
    inter  = max(0, x2 - x1) * max(0, y2 - y1)
    aire_a = (xa2 - xa1) * (ya2 - ya1)
    aire_b = (xb2 - xb1) * (yb2 - yb1)
    union  = aire_a + aire_b - inter
    return inter / union if union > 0 else 0.0


def verifier_worker(boite_worker, detections_frame, seuil_iou=0.1):
    """Vérifie la conformité d'un travailleur.
    Pour chaque règle, on cherche d'abord les boîtes de la partie_corps qui
    chevauchent le worker, puis on vérifie si l'EPI couvre cette partie.
    """
    motifs = []
    for classe_epi, regle in REGLES_CONFORMITE.items():
        partie = regle['partie_corps']
        # Parties du corps appartenant à ce worker (chevauchement avec sa boîte person)
        boites_partie = [
            d['boite'] for d in detections_frame
            if d['classe'] == partie and iou(boite_worker, d['boite']) > seuil_iou
        ]
        # Si aucune partie détectée pour ce worker, on ne peut pas conclure → on passe
        if not boites_partie:
            continue
        boites_epi = [d['boite'] for d in detections_frame if d['classe'] == classe_epi]
        for boite_partie in boites_partie:
            protege = any(iou(boite_partie, b) > seuil_iou for b in boites_epi)
            if not protege:
                motifs.append(regle['message'])
    return len(motifs) > 0, list(dict.fromkeys(motifs))

## 2. Pipeline de tracking

In [ ]:
COULEURS_ID = [
    (255, 87, 51), (51, 255, 87), (51, 87, 255), (255, 214, 51),
    (214, 51, 255), (51, 214, 255), (255, 51, 214), (150, 255, 51),
]

def couleur_id(track_id):
    return COULEURS_ID[int(track_id) % len(COULEURS_ID)]


def analyser_video_tracking(chemin_entree, chemin_sortie,
                             conf_min=0.4, seuil_iou=0.1,
                             fenetre=15, seuil_alerte=0.5):

    cap    = cv2.VideoCapture(str(chemin_entree))
    largeur = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    hauteur = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps     = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(chemin_sortie), fourcc, fps, (largeur, hauteur))

    # Historique de conformité par track_id
    historiques = defaultdict(lambda: deque(maxlen=fenetre))

    n_frames = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # Tracking ByteTrack natif YOLOv8
        resultats = model.track(frame, conf=conf_min, device=DEVICE,
                                persist=True, verbose=False)[0]

        # Extraire toutes les détections (pour la règle IoU)
        detections = []
        if resultats.boxes is not None:
            for boite, cls, conf in zip(
                resultats.boxes.xyxy.cpu().numpy(),
                resultats.boxes.cls.cpu().numpy(),
                resultats.boxes.conf.cpu().numpy(),
            ):
                detections.append({
                    'classe':    model.names[int(cls)],
                    'boite':     boite.tolist(),
                    'confiance': float(conf),
                })

        # Extraire les workers trackés (classe 'person')
        workers = []
        if resultats.boxes is not None and resultats.boxes.id is not None:
            for boite, cls, conf, tid in zip(
                resultats.boxes.xyxy.cpu().numpy(),
                resultats.boxes.cls.cpu().numpy(),
                resultats.boxes.conf.cpu().numpy(),
                resultats.boxes.id.cpu().numpy(),
            ):
                if model.names[int(cls)] == 'person':
                    workers.append({
                        'id':        int(tid),
                        'boite':     boite.tolist(),
                        'confiance': float(conf),
                    })

        img = frame.copy()
        h, w = img.shape[:2]
        alerte_globale = False

        # Dessiner les détections non-person
        for det in detections:
            if det['classe'] == 'person' or det['confiance'] < conf_min:
                continue
            x1, y1, x2, y2 = map(int, det['boite'])
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 1)
            cv2.putText(img, f"{det['classe']} {det['confiance']:.2f}",
                        (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 200, 0), 1)

        # Traiter chaque worker
        for worker in workers:
            tid   = worker['id']
            boite = worker['boite']
            x1, y1, x2, y2 = map(int, boite)
            couleur = couleur_id(tid)

            non_conforme_brut, motifs = verifier_worker(boite, detections, seuil_iou)
            historiques[tid].append(non_conforme_brut)

            non_conforme_lisse = (
                sum(historiques[tid]) / len(historiques[tid])
            ) >= seuil_alerte

            if non_conforme_lisse:
                alerte_globale = True
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                label = f"ID {tid} NON CONFORME"
                cv2.putText(img, label, (x1, max(y1 - 6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            else:
                cv2.rectangle(img, (x1, y1), (x2, y2), couleur, 2)
                cv2.putText(img, f"ID {tid}", (x1, max(y1 - 6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, couleur, 2)

        # Bandeau global si au moins un worker non-conforme
        if alerte_globale:
            cv2.rectangle(img, (0, 0), (w, 36), (0, 0, 255), -1)
            cv2.putText(img, 'ALERTE : travailleur non-conforme detecte', (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        writer.write(img)
        n_frames += 1

    cap.release()
    writer.release()
    print(f'{n_frames} frames traitées')
    print(f'Travailleurs trackés : {len(historiques)} IDs uniques')
    print(f'Vidéo sauvegardée    : {chemin_sortie}')

## 3. Application sur la vidéo de test

In [ ]:
VIDEO_SOURCE = Path('/root/Projet_Image/inference_output') / 'video_source.mp4'
VIDEO_SORTIE = OUTPUT_DIR / 'video_tracking.mp4'

analyser_video_tracking(
    VIDEO_SOURCE, VIDEO_SORTIE,
    conf_min     = 0.25,   # plus bas qu'en image fixe — EPI souvent détectés à conf ~0.3 en vidéo
    seuil_alerte = 0.7,    # flagge non-conforme seulement si EPI absent dans 70% des frames
)